In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = '../data/Sustainability_report_2024_kr.pdf'

print("문서 로드중")
loader = PyPDFLoader(file_path)
docs = loader.load()
print(len(docs))

문서 로드중
83


In [3]:
# 2. 청크 나누기
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = splitter.split_documents(docs[:20])
print(len(chunks))

48


In [5]:
# # 3. 벡터 스토어 
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

embeddings = OpenAIEmbeddings()
db_path = "../vectorstore/rag_eval_20"
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = db_path,
    collection_name = "samsung2024_eval"
)

In [6]:
# 벡터저장소가 이미 있는 상황

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
persist_dir = "../vectorstore/rag_eval_20"
collection_name = "samsung2024_eval"

vectorsore = Chroma(
    persist_directory=persist_dir,
    collection_name=collection_name,
    embedding_function=embedding
)

In [7]:
# 4. retriever 구성하기
retriever = vectorstore.as_retriever(
    search_kwargs = {"k" : 5}
)

In [8]:
# 5. 프롬프트 구성하기
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a helpful assistant. Answer strictly based on the provided context. "
    "If the answer is not in the context, say you don't know."
    "context : {context}"
)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system" , system_prompt),
    ("human", "{question}")
])

In [9]:
# 6. 모델 구성하기
model = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature = 0
)

In [10]:
# 7. 아웃풋 파서
from langchain_core.output_parsers import StrOutputParser
outputparser = StrOutputParser()

In [11]:
# 8. 체인 설정
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 문서 합치는 함수 설정
def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

# 체인만들기
rag_chain = (
    {"context" : RunnableLambda(lambda x : x["question"]) | retriever | format_docs,
     "question" :RunnablePassthrough()
     }
     | rag_prompt
     | model
     | outputparser
)

In [12]:
rag_chain.invoke({"question" :"삼성전자의 전망은?"})

'삼성전자는 어려운 대내외 환경 속에서도 지속 성장을 위해 역대 최고 수준의 연구개발 투자(28.3조원)와 전략적 시설투자(53.1조원)를 통해 기술 리더십을 강화하고 중장기 수요에 대응하고 있습니다. 또한, 글로벌 공시규제 프레임워크에 맞춰 지속가능경영 이슈를 발굴하고 관리 체계와 전략, 이행 활동을 충실히 수행하며, 환경, 사회, 경제적 리스크와 지정학적 불확실성 속에서도 지속가능성을 사업경쟁력과 기술혁신의 원동력으로 삼아 새로운 도약과 발전을 모색하고 있습니다. 따라서 삼성전자는 지속가능성을 기반으로 한 기술 혁신과 경영 활동을 통해 앞으로도 지속적인 성장과 발전을 기대하고 있습니다.'

In [ ]:
# 9. 평가용 데이터 불러오기
import pandas as pd

csv_path = "../data/report_2024_text.csv"
df = pd.read_csv(csv_path)
df.head()

,user_input,reference_contexts,reference,synthesizer_name
0,What initiatives has the company undertaken in...,"[""지속가능한 미래를 위한 노력을 계속해 \n왔습니다. 2050년 탄소중립을 통해 ...","In 인도, the company has transitioned 100% of th...",single_hop_specific_query_synthesizer
1,What are the core values established by 삼성전자 f...,['삼성전자 지속가능경영보고서 2024\n05\nOur Company Appendi...,"삼성전자는 경영철학을 반영한 5가지 핵심가치를 수립하였고, 이를 세부원칙과 행동지침...",single_hop_specific_query_synthesizer
2,How does DRAM fit into Samsung Electronics' bu...,['Our Company AppendixMateriality Assessment F...,DRAM is part of Samsung Electronics' Device So...,single_hop_specific_query_synthesizer
3,What are the financial results for SDC in 2023?,"['매출\n169조 9,923억 원\n영업이익\n14조 3,847억 원 네트워크\n...",The financial results for SDC in 2023 indicate...,single_hop_specific_query_synthesizer
4,삼성전자는 어떻게 이해관계자와 소통하나요?,['삼성전자 지속가능경영보고서 2024\n06\nOur Company Appendi...,삼성전자는 이해관계자와의 커뮤니케이션을 위해 지속가능경영 웹사이트를 통해 관련 정보...,single_hop_specific_query_synthesizer


- user_input : 질문
- reference_contexts : 예상되는 답변을 만들기위해 참고한 contexts
- reference : 예상되는 답변
***
- user_input : 질문
- retrieved_contexts : 검색한 자료 -> 리스트
- response : 실제 답변

In [16]:
# for 문으로 구현하기
answer = []
contexts = []
for question in df['user_input']:
    docs = retriever.invoke(question)
    ctx_list = []
    for d in docs:
        ctx_list.append(d.page_content)
    contexts.append(ctx_list)

In [17]:
contexts

[['A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024',
  '· 업종 간 협력\n·  기후 대응을 포함한  \nUN SDGs에 대한 기여\n·  투명하고 신속한 정보 공개\n·  기업 간담회\n·  NGO 미팅\n·  이해관계자 포럼\n·  시민사회 - 경영진 간담회\n·  노동인권 이해관계자 워크샵\n·  지속가능경영 웹사이트\n· 산업협회\n·  글로벌 NGO 대상 의견 수렴\n·  RBA\n1)\n, RMI\n2)\n, BSR\n3)\n 활동\n1) Responsible Business Alliance\n2) Responsible Minerals Initiative\n3) Business for Social Responsibility\n·  UNGC\n1)\n·  ACEC\n2)\n, SCC\n3)\n 활동\n1) United Nations Global Compact\n2) Asia Clean Energy Coalition\n3) Semiconductor Climate Consortium\n정부 ·  간접 경제효과(투자, 고용 등 파생효과)\n·  공정거래\n·  안전·보건\n·  컴플라이언스\n·  기업윤리\n·  정책 간담회\n·  국회\n·  정책수립 공청회\n·  정책자문기구\n·  지속가능경영 웹사이트\n·  정부와 협업하여 중소기업 지원 프로그램 운영 ·  정부와 협업하여 벤처투자 창구 설립·운영\n언론 ·  주요 제품/사업 실적 및 전략\n·  투자, R&D, M&A,  \n신사업 등 미래 성장 전략 \n·  탄소중립 등 ESG 추진 성과\n·  인/노사, 환경안전, 특허,  \n제품·서비스 품질 등\n·  보도자료\n·  지속가능경영 웹사이트\n·  삼성전자 반도체 뉴스룸\n·  삼성전자 뉴스룸\n· 미디어 간담회\n·  글로벌 IT 전시회·신제품 발표 취재지원\n·  미디어데이 개최\n·  기자회견\n·  인터뷰\n·  기획홍보\n Our Company',
  'A Jour

In [18]:
for question in df['user_input']:
    an = rag_chain.invoke({"question" : question})
    answer.append(an)

In [19]:
answer[:5]

['The company has included India as one of the global major manufacturing sites where the DX division has achieved 100% renewable energy use for electricity consumption as part of its goal to achieve carbon neutrality by 2030. This initiative contributes to sustainable development by reducing carbon emissions and promoting the use of renewable energy sources in India.',
 '삼성전자는 지속가능경영을 위해 5가지 핵심가치를 수립하였으며, 이 핵심가치를 세부원칙과 행동지침으로 구체화하여 글로벌 행동규범(Global Code of Conduct)을 제정하였습니다. 다만, 구체적인 5가지 핵심가치의 내용은 제공된 문서 내에 명시되어 있지 않습니다.',
 "DRAM is part of Samsung Electronics' DS (Device Solutions) business division, which focuses on semiconductor components. Specifically, DS부문 includes the memory semiconductor business, Foundry business, and System LSI business, producing and selling components such as DRAM, NAND Flash, and mobile AP.",
 'The financial results for SDC in 2023 are not provided in the given context. The sales and operating profit mentioned exclude SDC and Harman.',
 '삼성전자는 다양한 이해관계자와 소

In [26]:
df['response'] = answer
df['retrieved_contexts'] = contexts
df.head()

,user_input,reference_contexts,reference,synthesizer_name,response,retrieved_contexts
0,What initiatives has the company undertaken in...,"[""지속가능한 미래를 위한 노력을 계속해 \n왔습니다. 2050년 탄소중립을 통해 ...","In 인도, the company has transitioned 100% of th...",single_hop_specific_query_synthesizer,The company has included India as one of the g...,[A Journey Towards \na Sustainable Future\n삼성...
1,What are the core values established by 삼성전자 f...,['삼성전자 지속가능경영보고서 2024\n05\nOur Company Appendi...,"삼성전자는 경영철학을 반영한 5가지 핵심가치를 수립하였고, 이를 세부원칙과 행동지침...",single_hop_specific_query_synthesizer,"삼성전자는 지속가능경영을 위해 5가지 핵심가치를 수립하였으며, 이 핵심가치를 세부원...",[A Journey Towards \na Sustainable Future\n삼성...
2,How does DRAM fit into Samsung Electronics' bu...,['Our Company AppendixMateriality Assessment F...,DRAM is part of Samsung Electronics' Device So...,single_hop_specific_query_synthesizer,DRAM is part of Samsung Electronics' DS (Devic...,[삼성전자 지속가능경영보고서 2024\n05\nOur Company Appendix...
3,What are the financial results for SDC in 2023?,"['매출\n169조 9,923억 원\n영업이익\n14조 3,847억 원 네트워크\n...",The financial results for SDC in 2023 indicate...,single_hop_specific_query_synthesizer,The financial results for SDC in 2023 are not ...,[DS Device Solutions\n메모리\n※ 상기 매출과 영업이익은 2023...
4,삼성전자는 어떻게 이해관계자와 소통하나요?,['삼성전자 지속가능경영보고서 2024\n06\nOur Company Appendi...,삼성전자는 이해관계자와의 커뮤니케이션을 위해 지속가능경영 웹사이트를 통해 관련 정보...,single_hop_specific_query_synthesizer,삼성전자는 다양한 이해관계자와 소통하기 위해 지속가능경영 웹사이트를 통해 관련 정보...,[기회 등에 대해 논의하였습니다. 삼성전자는 참 석한 이해관계자들과의 미팅을 \n통...


In [ ]:
df.to_csv("../data/report_2024_answer.csv", index = False)

In [39]:
df = pd.read_csv("../data/report_2024_answer.csv")

In [ ]:
from ast import literal_eval
df['reference_contexts'] = df['reference_contexts'].apply(lambda x: literal_eval(x))
df['retrieved_contexts'] = df['retrieved_contexts'].apply(lambda x: literal_eval(x))

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   user_input          100 non-null    object
 1   reference_contexts  100 non-null    object
 2   reference           100 non-null    object
 3   synthesizer_name    100 non-null    object
 4   response            100 non-null    object
 5   retrieved_contexts  100 non-null    object
dtypes: object(6)
memory usage: 4.8+ KB


In [47]:
type(df['retrieved_contexts'].iloc[0])

list

In [49]:
type(df['reference'].iloc[0])

str

In [48]:
type(df['reference_contexts'].iloc[0])

list

In [44]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, LLMContextRecall, FactualCorrectness
from ragas.llms import LangchainLLMWrapper

# ragas 평가
eval_llm = LangchainLLMWrapper(model)
dataset = EvaluationDataset.from_pandas(df)
dataset

C:\Users\user\AppData\Local\Temp\ipykernel_29936\3001053187.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  eval_llm = LangchainLLMWrapper(model)


EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference'], len=100)

In [50]:
scores = evaluate(
    dataset,
    metrics = [Faithfulness(), LLMContextRecall(), FactualCorrectness()],
    llm = eval_llm
)

Evaluating:   0%|          | 0/300 [00:00<?, ?it/s]

Exception raised in Job[71]: TimeoutError()
Exception raised in Job[259]: TimeoutError()
Exception raised in Job[281]: TimeoutError()
Exception raised in Job[298]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}})
Exception raised in Job[296]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}})
Exception raised in Job[285]: TimeoutError()
Exception raised in Job[293]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your cur

In [51]:
scores

{'faithfulness': 0.9066, 'context_recall': 0.7641, 'factual_correctness(mode=f1)': 0.4110}